# 01 Dataset inventory and acquisition
Loads the Phase 1 manifests and re-hashes one small file independently to confirm the checksum manifest. Evidence status of each statement is in `reports/dataset_provenance.md`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
R = ROOT / "results/audit"
M = ROOT / "data/manifests"
pd.set_option("display.max_rows", 80); pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("project root:", ROOT.name)

project root: research_paper


In [2]:
inv = pd.read_csv(M / 'dataset_inventory.csv')
inv.groupby(['record_type','dataset_family','variant'], dropna=False)['size_bytes'].agg(['count','sum'])

count          sum
record_type    dataset_family   variant                          
archive_member CIC-IDS-2017     improved           5   1152270765
               CSE-CIC-IDS-2018 improved          10  36044466483
file           CIC-IDS-2017     improved           6   1495819778
                                original           8   1203105382
               CSE-CIC-IDS-2018 improved          11  46471318212
                                original          10   6886649507
               both             documentation      1       270785
               NaN              NaN                1        34530

In [3]:
ver = pd.read_csv(M / 'acquisition_verification.csv')
print('files verified after extraction/copy:', int(ver['verified'].sum()), 'of', len(ver))
ver[['dest_relpath','size_bytes','verified','n_newline_bytes']]

files verified after extraction/copy: 33 of 33


,dest_relpath,size_bytes,verified,n_newline_bytes
0,data/raw_improved/CIC-IDS-2017/friday.csv,285188226,True,547558
1,data/raw_improved/CIC-IDS-2017/monday.csv,207875155,True,371625
2,data/raw_improved/CIC-IDS-2017/thursday.csv,189519159,True,362077
3,data/raw_improved/CIC-IDS-2017/tuesday.csv,178397720,True,322079
4,data/raw_improved/CIC-IDS-2017/wednesday.csv,291290505,True,496642
5,data/raw_improved/CSE-CIC-IDS-2018/Friday-02-0...,3689837784,True,6311372
6,data/raw_improved/CSE-CIC-IDS-2018/Friday-16-0...,4211741805,True,7390267
7,data/raw_improved/CSE-CIC-IDS-2018/Friday-23-0...,3409376619,True,5976482
8,data/raw_improved/CSE-CIC-IDS-2018/Thursday-01...,3808397458,True,6551402
9,data/raw_improved/CSE-CIC-IDS-2018/Thursday-15...,2992535617,True,5410103


In [4]:
s3 = pd.read_csv(M / 'original_2018_s3_etag_check.csv')
s3[['relpath','size_match','s3_etag','etag_match','s3_last_modified']]

,relpath,size_match,s3_etag,etag_match,s3_last_modified
0,data/raw_original/CSE-CIC-IDS-2018/Friday-02-0...,True,2cef2d9c87fc74df84d7424f6949f1b3-22,True,"Thu, 11 Oct 2018 16:02:49 GMT"
1,data/raw_original/CSE-CIC-IDS-2018/Friday-16-0...,True,e68ef27c09c98ba91fe3c6b1108a18b9-20,True,"Thu, 11 Oct 2018 16:03:10 GMT"
2,data/raw_original/CSE-CIC-IDS-2018/Friday-23-0...,True,472d63fb3c1285ce2f1e25617ba8cba5-23,True,"Thu, 11 Oct 2018 16:03:33 GMT"
3,data/raw_original/CSE-CIC-IDS-2018/Thuesday-20...,True,49075d3ee07bcd47e1aae3f059953799-242,True,"Thu, 11 Oct 2018 16:03:59 GMT"
4,data/raw_original/CSE-CIC-IDS-2018/Thursday-01...,True,5889e4b7e0a421070747a2441a2772d7-7,True,"Thu, 11 Oct 2018 16:08:38 GMT"
5,data/raw_original/CSE-CIC-IDS-2018/Thursday-15...,True,eb1dc2ec76efe09c8995f28bfa4c8656-23,True,"Thu, 11 Oct 2018 16:08:48 GMT"
6,data/raw_original/CSE-CIC-IDS-2018/Thursday-22...,True,2bfbfdc038eb59f60bb725d9f3ede5d7-23,True,"Thu, 11 Oct 2018 16:09:20 GMT"
7,data/raw_original/CSE-CIC-IDS-2018/Wednesday-1...,True,46c0f45f8e6fe1edf1f08487448c102f-22,True,"Thu, 11 Oct 2018 16:09:44 GMT"
8,data/raw_original/CSE-CIC-IDS-2018/Wednesday-2...,True,3a1d6b804ba9f6a3085973c2fa425883-20,True,"Thu, 11 Oct 2018 16:10:12 GMT"
9,data/raw_original/CSE-CIC-IDS-2018/Wednesday-2...,True,af10d2362a17be6331cfc6d1ce2474f2-13,True,"Thu, 11 Oct 2018 16:10:33 GMT"


In [5]:
# independent re-hash of one improved file, compared with the checksum manifest
from src.audit.inventory import sha256_stream, load_checksums
p = ROOT / 'data/raw_improved/CIC-IDS-2017/friday.csv'
with p.open('rb') as fh:
    digest, n = sha256_stream(fh)
ck = load_checksums(M / 'checksums_sha256.csv')
expected = ck[('archive_member', 'CICIDS2017_improved.zip', 'friday.csv')]['sha256']
print(digest == expected, n)

True 285188226
